In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sms

from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from sklearn.model_selection import GridSearchCV

In [2]:
df = pd.read_csv('insurance.csv')
df.shape

(1338, 7)

In [3]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [5]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [6]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [7]:
df['sex']=df['sex'].map({'female':1,'male':0})
df['smoker']=df['smoker'].map({'yes':1,'no':0})

In [8]:
df = pd.get_dummies(df,drop_first=True)

In [9]:
df.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,1,27.900,0,1,16884.92400,False,False,True
1,18,0,33.770,1,0,1725.55230,False,True,False
2,28,0,33.000,3,0,4449.46200,False,True,False
3,33,0,22.705,0,0,21984.47061,True,False,False
4,32,0,28.880,0,0,3866.85520,True,False,False


In [10]:
X = df.drop('charges',axis=1)
y = df['charges']

In [11]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [12]:
scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [13]:
model1 = SVR(kernel='linear')
model1.fit(X_train,y_train)
y_pred1 = model1.predict(X_test)
print("MSE:",mean_squared_error(y_test,y_pred1))
print("MAE:",mean_absolute_error(y_test,y_pred1))
print("r2_score",r2_score(y_test,y_pred1))

MSE: 152174927.58241966
MAE: 8038.31445677021
r2_score 0.019799220771840598


In [14]:
model2 = SVR(kernel='rbf')
model2.fit(X_train,y_train)
y_pred2 = model2.predict(X_test)
print("MSE:",mean_squared_error(y_test,y_pred2))
print("MAE:",mean_absolute_error(y_test,y_pred2))
print("r2_score",r2_score(y_test,y_pred2))

MSE: 166128803.80848217
MAE: 8612.408423351833
r2_score -0.07008155372454805


In [15]:
model3 = SVR(kernel='poly')
model3.fit(X_train,y_train)
y_pred3 = model3.predict(X_test)
print("MSE:",mean_squared_error(y_test,y_pred3))
print("MAE:",mean_absolute_error(y_test,y_pred3))
print("r2_score",r2_score(y_test,y_pred3))

MSE: 165713134.46771246
MAE: 8607.801381076031
r2_score -0.0674041125836411


In [16]:
param_grid = {
    'kernel':['rbf','linear'],
    'C':[1,10,100,1000],
    'epsilon':[0.01,0.1,1],
    'gamma':['scale','auto']
}
grid = GridSearchCV(SVR(),param_grid,cv=5,scoring='r2')
grid.fit(X_train,y_train)
print(grid.best_params_)
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
print("R2 Score:",r2_score(y_test,y_pred))

{'C': 1000, 'epsilon': 1, 'gamma': 'auto', 'kernel': 'rbf'}
R2 Score: 0.7468897898826761


In [17]:
param_grid = {
    'kernel': ['linear','rbf','poly'],
    'C': [0.1, 1, 10, 100],
    'epsilon': [0.1, 0.2, 0.5, 1],
    'gamma': ['scale','auto']
}
svr = SVR()
grid = GridSearchCV(
    estimator=svr,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
grid.fit(X_train,y_train)
print(grid.best_params_)
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
print("R2 Score:",r2_score(y_test,y_pred))

{'C': 100, 'epsilon': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
R2 Score: 0.6507722021251104
